# Stage 3 — Mine the predicate edges (frozen text; two rules)

Reads the **frozen** extracted text in `data/text/` (recorded in `TEXT_SNAPSHOT.json` by
`freeze_text.py`) and applies **both** deterministic edge rules from `edge_rules.py`:

* **Rule 1 (registered, prereg §5):** in any summary containing a predicate cue, every other
  in-corpus K-number mentioned anywhere becomes an edge. Written to `data/predicate_edges_rule1.csv`.
* **Rule 2 (restricted; DEVIATIONS.md D4):** Rule 1 minus K-numbers whose every mention sits inside a
  *component/accessory compatibility* section or table, or an *FDA reference device* section.
  Written to `data/predicate_edges.csv` — this is the **primary** edge set Stages 6–8 consume.

Every edge Rule 2 removes is listed in `data/excluded_edges.csv` with the heading that excluded it.
Each edge is labelled `NEAR_CUE` (strong cue within 300 characters of the K-number's first counted
mention) or `DISTANT_CUE`. These were previously named `SECTION_HEADED` / `PROXIMITY_ONLY`; the rule
is unchanged, only the names, which no longer imply section detection.

Network: none. **This stage does not open PDFs** — if a text file is missing, restore `data/text/`
from OSF rather than re-extracting (D5).

In [1]:
import os, json, random
import pandas as pd
import edge_rules as er
from freeze_text import hash_corpus, retrieved_knumbers

corpus = pd.read_csv("data/corpus.csv", dtype=str)
VALID = set(corpus["k_number"])
have, status_counts = retrieved_knumbers()
print(f"{len(have)} retrieved summaries in the manifest  (status counts {status_counts})")

12388 retrieved summaries in the manifest  (status counts {'404': 924, '200': 12388, '-1': 1})


### Verify the frozen text corpus before using it

In [2]:
frozen = json.load(open("TEXT_SNAPSHOT.json"))
_, corpus_hash, missing = hash_corpus(have)
assert not missing, f"{len(missing)} text files missing from data/text/ — restore from OSF (first: {missing[:5]})"
assert corpus_hash == frozen["corpus_sha256"], "data/text/ does not match TEXT_SNAPSHOT.json — restore from OSF"
print(f"CHECKPOINT  frozen text corpus OK: {frozen['n_files']} files, frozen {frozen['frozen_on']}, "
      f"sha256 {corpus_hash[:16]}…")

def get_text(k):
    return open(f"data/text/{k}.txt", encoding="utf-8").read()

CHECKPOINT  frozen text corpus OK: 12388 files, frozen 2026-09-16, sha256 27646debf7d7176c…


### Apply both rules

In [3]:
rule1, rule2, excluded, coverage = [], [], [], []
for k in have:
    r = er.extract(get_text(k), k, VALID)
    for pred, conf in r["rule1"]:
        rule1.append({"predicate_knumber": pred, "device_knumber": k, "confidence": conf})
    for pred, conf in r["rule2"]:
        rule2.append({"predicate_knumber": pred, "device_knumber": k, "confidence": conf})
    for pred, kind, heading, conf in r["excluded"]:
        excluded.append({"predicate_knumber": pred, "device_knumber": k, "zone": kind,
                         "zone_heading": heading, "rule1_confidence": conf})
    # coverage diagnostic (per document)
    if not r["has_cue"]:
        cov = "no_cue"
    elif r["rule2"]:
        cov = "edge_resolvable"
    elif r["rule1"]:
        cov = "excluded_only"        # NEW: every in-corpus K-number sat in an exclusion zone
    elif r["found"]:
        cov = "out_of_scope"         # cites K-numbers, but none in our corpus
    else:
        cov = "name_only"            # cue present, but no K-number at all
    coverage.append({"k_number": k, "coverage": cov,
                     "coverage_rule1": "edge_resolvable" if r["rule1"] else cov})

e1 = pd.DataFrame(rule1).drop_duplicates()
e2 = pd.DataFrame(rule2).drop_duplicates()
ex = pd.DataFrame(excluded).drop_duplicates(subset=["predicate_knumber", "device_knumber"])
cov = pd.DataFrame(coverage)

e1.to_csv("data/predicate_edges_rule1.csv", index=False)
e2.to_csv("data/predicate_edges.csv", index=False)          # PRIMARY (Rule 2)
ex.to_csv("data/excluded_edges.csv", index=False)
cov.to_csv("data/coverage_diagnostic.csv", index=False)

n = len(have); cc = cov["coverage"].value_counts()
print(f"CHECKPOINT  Rule 1 edges {len(e1)} | devices with >=1 edge {e1['device_knumber'].nunique()}")
print(f"CHECKPOINT  Rule 2 edges {len(e2)} | devices with >=1 edge {e2['device_knumber'].nunique()}")
print(f"CHECKPOINT  excluded by Rule 2: {len(ex)} edges "
      f"({100*len(ex)/max(1,len(e1)):.1f}% of Rule 1) in {ex['device_knumber'].nunique()} documents "
      f"| by zone {ex['zone'].value_counts().to_dict()}")
print(f"CHECKPOINT  Rule 2 confidence {e2['confidence'].value_counts().to_dict()}")
print(f"CHECKPOINT  coverage (Rule 2): cue {100*(n-cc.get('no_cue',0))/n:.1f}% | "
      f"resolvable {100*cc.get('edge_resolvable',0)/n:.1f}% | excluded-only {100*cc.get('excluded_only',0)/n:.2f}% | "
      f"name-only {100*cc.get('name_only',0)/n:.1f}% | out-of-scope {100*cc.get('out_of_scope',0)/n:.1f}%")

CHECKPOINT  Rule 1 edges 30476 | devices with >=1 edge 8978
CHECKPOINT  Rule 2 edges 29234 | devices with >=1 edge 8942
CHECKPOINT  excluded by Rule 2: 1242 edges (4.1% of Rule 1) in 538 documents | by zone {'reference': 920, 'compat': 322}
CHECKPOINT  Rule 2 confidence {'NEAR_CUE': 21541, 'DISTANT_CUE': 7693}
CHECKPOINT  coverage (Rule 2): cue 87.5% | resolvable 72.2% | excluded-only 0.29% | name-only 2.8% | out-of-scope 12.2%


### Validation sample (D3) — same 60 devices, seed 42, now under both rules

`data/validation_sample.csv` holds the Rule 2 edges for the 60 sampled devices;
`data/validation_sample_rule1.csv` the Rule 1 edges for the same devices, so the before/after
precision is measured on identical documents.

In [4]:
random.seed(42)
sample = random.sample(have, min(60, len(have)))
e2[e2["device_knumber"].isin(sample)].to_csv("data/validation_sample.csv", index=False)
e1[e1["device_knumber"].isin(sample)].to_csv("data/validation_sample_rule1.csv", index=False)
print(f"CHECKPOINT  validation sample: 60 devices | Rule 1 edges {e1['device_knumber'].isin(sample).sum()} "
      f"| Rule 2 edges {e2['device_knumber'].isin(sample).sum()}")

CHECKPOINT  validation sample: 60 devices | Rule 1 edges 121 | Rule 2 edges 120
